# Phase 3: diagnosing PaDiM's inverse step on the GPU

**CRISP-DM stage:** Data preparation

**Why this exists:** PaDiM crashed twice on the Kaggle GPU while inverting its tables of numbers. The log names `magma_sgetrf_batched`. MAGMA is a maths library PyTorch uses on GPUs, `sgetrf` is the first stage of inverting a table, and `batched` means many tables at once.

**Runs on:** Kaggle with the GPU on and internet off. On the laptop it only checks that the code runs, because the laptop has no GPU and cannot reproduce the crash.

**Uses made-up numbers only:** no dataset, no anomalib and no pretrained network. The tables are built to resemble PaDiM's, but this is a stand-in, not proof about the real tables.

## 1. Versions

The cause may depend on these, so they are recorded. If PyTorch differs from the failed runs (2.11.0 built for CUDA 13.0), the comparison is weaker.

In [ ]:
import subprocess
import sys

import torch

version_lines = []
version_lines.append("python: " + sys.version.split()[0])
version_lines.append("torch: " + torch.__version__)
version_lines.append("torch built for cuda: " + str(torch.version.cuda))
version_lines.append("processor threads: " + str(torch.get_num_threads()))

if torch.cuda.is_available():
    version_lines.append("gpu: " + torch.cuda.get_device_name(0))
    version_lines.append("gpu memory GB: " + str(round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)))
    driver = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                            capture_output=True, text=True)
    version_lines.append("driver: " + driver.stdout.strip())
else:
    version_lines.append("gpu: none")

with open("versions.txt", "w") as versions_file:
    for line in version_lines:
        versions_file.write(line + "\n")

for line in version_lines:
    print(line)

## 2. The test script

A GPU error can jam the GPU for the rest of a Python session. So this cell writes the tests to a file, and part 3 runs each part in its own fresh Python session.

**Made-up tables:** `random × its own flip`, divided by 7, gives a symmetric table, the same kind PaDiM builds from 8 photos, since 8 photos leave 7 independent differences. PaDiM's two small additions along the diagonal (0.01 and 0.00001) are copied, because they are what make its tables invertible. A fixed seed makes them repeatable.

**Sizes:** 100 and 550 are the feature counts anomalib uses for ResNet-18 and Wide ResNet-50. 4096 is the number of positions at image size 256.

**Accuracy:** a table times its inverse should give the identity table, so the largest difference shows whether an answer is right. It is checked on the first 16 tables only, to save memory and time.

**Stopping:** a GPU error stops the part immediately. Running out of memory only ends that table size, because larger counts would fail too, and it is a limit, not a crash.

**Results are saved after every test,** so a crash cannot lose them. Results are not kept on the GPU between groups, so a real fit, which must keep them, would need more memory than shown.

In [ ]:
SCRIPT_TEXT = r'''import csv
import sys
import time

import torch

RESULTS_FILE = "inverse_results.csv"
SEED = 42
PHOTO_DIFFERENCES = 7
# PaDiM adds these two values along the diagonal of each table
DIAGONAL_ADDITION = 0.01 + 0.00001
TABLES_CHECKED_FOR_ACCURACY = 16

if torch.cuda.is_available():
    TABLE_SIZES = [100, 550]
    COUNTS = [1, 16, 256, 1024, 4096]
    FULL_COUNT = 4096
    GROUPS = {100: [1024, 256, 64, 16, 1], 550: [1024, 256, 64]}
else:
    # The laptop only checks that the code runs
    TABLE_SIZES = [10, 20]
    COUNTS = [1, 4]
    FULL_COUNT = 8
    GROUPS = {10: [4, 1], 20: [4]}


def make_tables(count, size):
    """Build made-up symmetric tables resembling PaDiM's, repeatable with the seed."""
    generator = torch.Generator().manual_seed(SEED)
    random_numbers = torch.randn(count, size, PHOTO_DIFFERENCES, generator=generator)
    tables = torch.bmm(random_numbers, random_numbers.transpose(1, 2)) / PHOTO_DIFFERENCES
    return tables + DIAGONAL_ADDITION * torch.eye(size)


def wait_for_gpu(device):
    """Wait for the GPU to finish, so the timer does not stop early."""
    if device == "cuda":
        torch.cuda.synchronize()


def largest_error(tables, inverses):
    """Return how far table times inverse is from the identity table."""
    product = torch.bmm(tables, inverses)
    identity = torch.eye(tables.shape[1], device=tables.device)
    return float((product - identity).abs().max())


def record(part, size, count, group, outcome, seconds, peak_gb, error):
    """Save one test's result straight away, and print it."""
    with open(RESULTS_FILE, "a", newline="") as results_file:
        csv.writer(results_file).writerow([part, size, count, group, outcome, seconds, peak_gb, error])
    print(part, size, count, group, outcome, seconds, peak_gb, error, flush=True)


def run_test(part, size, count, group, device):
    """Invert the tables in groups. Return worked, out of memory or gpu error."""
    tables = make_tables(count, size)
    if device == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    error = ""
    try:
        wait_for_gpu(device)
        started = time.perf_counter()
        for start in range(0, count, group):
            group_tables = tables[start:start + group].to(device)
            inverses = torch.linalg.inv(group_tables)
            wait_for_gpu(device)
            if start == 0:
                checked = TABLES_CHECKED_FOR_ACCURACY
                first_error = largest_error(group_tables[:checked], inverses[:checked])
                error = round(first_error, 6)
        seconds = round(time.perf_counter() - started, 3)
        peak_gb = ""
        if device == "cuda":
            peak_gb = round(torch.cuda.max_memory_allocated() / 1e9, 2)
        record(part, size, count, group, "worked", seconds, peak_gb, error)
        return "worked"
    except torch.cuda.OutOfMemoryError:
        record(part, size, count, group, "out of memory", "", "", "")
        return "out of memory"
    except torch.AcceleratorError as gpu_error:
        message = str(gpu_error).split("\n")[0]
        record(part, size, count, group, "gpu error: " + message, "", "", "")
        return "gpu error"


def part_a():
    """Whole tables at once, from small up to PaDiM's real size."""
    for size in TABLE_SIZES:
        for count in COUNTS:
            outcome = run_test("A", size, count, count, "cuda")
            if outcome == "gpu error":
                return
            if outcome == "out of memory":
                break


def part_b():
    """The full number of tables, split into smaller groups."""
    for size in TABLE_SIZES:
        for group in GROUPS[size]:
            outcome = run_test("B", size, FULL_COUNT, group, "cuda")
            if outcome == "gpu error":
                return


def part_c():
    """The full number of tables on the processor, which does not use the GPU."""
    for size in TABLE_SIZES:
        run_test("C", size, FULL_COUNT, FULL_COUNT, "cpu")


part_name = sys.argv[1]
if part_name == "A":
    if torch.cuda.is_available():
        part_a()
elif part_name == "B":
    if torch.cuda.is_available():
        part_b()
else:
    part_c()
'''

with open("inverse_tests.py", "w") as script_file:
    script_file.write(SCRIPT_TEXT)
print("Test script written")

## 3. Run each part in a fresh session

- **Part A:** whole tables at once, from small up to real size. Where does it first fail?
- **Part B:** the full 4096 tables in smaller groups. Does splitting avoid the crash?
- **Part C:** the processor. Does it work there, and how long does it take?

`subprocess.run` starts a new Python session and waits for it. `timeout` stops a part if it runs too long, because every minute uses GPU hours. 6 minutes per part keeps the whole run under the 15-minute limit agreed before starting, with time left for start-up. Each part's messages, including any MAGMA errors, are saved to its own file.

In [ ]:
import csv

PART_TIME_LIMIT_SECONDS = 360

with open("inverse_results.csv", "w", newline="") as results_file:
    csv.writer(results_file).writerow(["part", "table_size", "tables", "group", "outcome", "seconds", "peak_gpu_gb", "largest_error"])

for part_name in ["A", "B", "C"]:
    print("=== Part", part_name)
    try:
        finished = subprocess.run([sys.executable, "inverse_tests.py", part_name],
                                  capture_output=True, text=True, timeout=PART_TIME_LIMIT_SECONDS)
        output = finished.stdout + "\n--- messages\n" + finished.stderr
        print(finished.stdout)
        print("Part", part_name, "exit code:", finished.returncode)
    except subprocess.TimeoutExpired as stopped:
        output = "Stopped at the time limit"
        with open("inverse_results.csv", "a", newline="") as results_file:
            csv.writer(results_file).writerow([part_name, "", "", "", "stopped at the time limit", "", "", ""])
        print("Part", part_name, "stopped at the time limit")
    with open("part_" + part_name + "_messages.txt", "w") as messages_file:
        messages_file.write(output)